# L27 · Aprendizaje continuo: alertar o reentrenar

**S27 · Lectura: Cap. 31 · 40–55 min**

## La pregunta

¿Qué señal pide reentrenar y cuál pide revisar la identificación?

## Qué harás

Aplicar una regla de decisión a cambios en datos y efectos.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Una política de continuar, alertar, abstenerse, revisar o reentrenar.

> **Idea clave:** Reentrenar no corrige un confusor nuevo ni un cambio en la pregunta causal.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
from lab_assumptions import asmd, psi, auc_score, estabilidad_efecto, ks_stat
import numpy as np, pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
rng = setup()
REPS = R(10, 40)

In [ ]:
# El monitoreo decide el siguiente paso
import matplotlib.pyplot as plt
labels = ['vigilar', 'alertar', 'abstenerse', 'revisar', 'reentrenar']
fig, ax = plt.subplots(figsize=(8, 2.4))
ax.plot(range(len(labels)), [0] * len(labels), "o-", lw=3)
for i, label in enumerate(labels):
    ax.text(i, 0.08, label, ha="center", va="bottom", fontsize=10)
ax.set_xlim(-.35, len(labels) - .65); ax.set_ylim(-.2, .3); ax.axis("off")
ax.set_title("El monitoreo decide el siguiente paso")
plt.tight_layout(); plt.show()

## 1. Las dos opciones y el vocabulario
| Opción | Qué hace | Riesgo en un sistema causal |
|---|---|---|
| Reentrenamiento automático periódico | El estimador se reajusta con los datos recientes en un calendario fijo | El nuevo estimador puede reflejar **otra población o otro mecanismo** sin que nadie lo audite; un placebo no lo detecta |
| **Champion-challenger con validación humana** (la que adopta el curso) | El reentrenado compite en paralelo; reemplaza solo si pasa un chequeo explícito que incluye **los supuestos de identificación** | Más lento y exige mantener dos modelos |

**Dos vocabularios.** El Cap. 31 dice «promover» y «revisión humana requerida»; el sílabo, en cambio, usa una lista cerrada. Esta es la correspondencia que usamos en el curso:

| Libro (Cap. 31) | Sílabo (lista cerrada) | Cuándo |
|---|---|---|
| (el champion sigue) | **continuar** | Todo pasa y el efecto no cambió |
| Revisión humana requerida | **alertar** | Falla la prueba estadística o cambió la población |
| Revisión humana requerida | **revisar identificación** | El mecanismo de asignación cambió: la no confusión ya no es defendible |
| — (el libro no lo cubre) | **abstenerse** | No hay soporte suficiente para estimar |
| promover | **reentrenar** | Doble validación superada y el efecto cambió de verdad |

> El ejemplo del §6.1 del sílabo usa «reestimar», que no está en la lista: debe leerse como **reentrenar** (C9).

## 2. Un mundo con verdad: tres escenarios
Ocho periodos. En el periodo 0 se entrena el champion. En **S1** aparece, desde el periodo 4, un **confusor no medido** (la asignación pasa a depender de $U$, que no está en los datos). En **S2**, desde el periodo 4, **cambia la población** (la covariable $X_2$ se desplaza) y el efecto depende de $X_2$. **S0** no cambia nada.

In [ ]:
P, CAMBIO, N_VENT = 8, 4, 800
def mundo(esc, periodo, n, g):
    cambia = periodo >= CAMBIO
    X1 = g.normal(size=n); X2 = g.normal(size=n) + (1.5 if esc == "S2" and cambia else 0.0); U = g.normal(size=n)
    z = (2.0 * U + 0.1 * X1) if (esc == "S1" and cambia) else 1.5 * X1
    D = g.binomial(1, 1 / (1 + np.exp(-z)))
    tau = 2.0 + (1.0 * X2 if esc == "S2" else 0.0 * X2)
    Y = tau * D + 1.0 * X1 + 0.5 * X2 + 1.0 * U + g.normal(size=n)
    return pd.DataFrame({"X1": X1, "X2": X2, "D": D, "Y": Y}), float(tau.mean())
COVS = ["X1", "X2"]
def estimar(df):
    """ATE de la ventana: Y ~ D + X + D·(X − media). El coeficiente de D es el efecto en la media de X."""
    x1c, x2c = df.X1 - df.X1.mean(), df.X2 - df.X2.mean()
    r = ols(df.Y.to_numpy(float), np.c_[df.D, df.X1, df.X2, df.D * x1c, df.D * x2c]); return float(r["coef"][1]), float(r["se"][1])
g0 = np.random.default_rng(0)
for esc in ("S0", "S1", "S2"):
    w, ate = mundo(esc, CAMBIO, 4000, g0); th, se = estimar(w)
    print(f"{esc}: tras el cambio, ATE verdadero de la ventana = {ate:.2f} · estimador reentrenado = {th:.2f} (EE {se:.2f})")

**Mira el resultado.** En S1 el ATE verdadero sigue siendo 2 pero el reentrenado dice otra cosa. ¿Por qué el reentrenamiento «hereda» el sesgo?

**Respuesta:**

Porque el estimador solo ajusta por lo que **mide** ($X_1$, $X_2$). Cuando la asignación pasa a depender de $U$, que además afecta a $Y$, queda un camino de puerta trasera abierto que ningún reajuste con los mismos datos cierra: el modelo nuevo es **más ajustado** a los datos nuevos y más sesgado. Las métricas de ajuste mejoran; la identificación se rompió.



## 3. El Listado 31.1 tal como está (E12)
Tres problemas: (1) `pasa_tests_estadisticos` está **fijo en `True`** («asumir que ya pasó CI/CD»): no puede fallar; (2) el `challenger` ni siquiera se usa; (3) compara con **p-valores de KS**, justo lo que el Cap. 30 desaconseja. Lo corrido, con la misma lógica:

In [ ]:
class Modelo:  # contenedor mínimo para el listado
    def __init__(self, datos): self.datos_entrenamiento = datos
class Esquema:
    covariates = COVS
def evaluar_challenger_libro(champion, challenger, df_reciente, schema, alpha=0.05):
    resultado = {"pasa_tests_estadisticos": True, "poblacion_comparable": True}
    for c in schema.covariates:
        _, p = stats.ks_2samp(champion.datos_entrenamiento[c], df_reciente[c])
        if p < alpha: resultado["poblacion_comparable"] = False
    resultado["promover"] = resultado["pasa_tests_estadisticos"] and resultado["poblacion_comparable"]
    if not resultado["promover"]: resultado["accion"] = "Revision humana requerida antes de promover"
    return resultado
prom = {}
for esc in ("S0", "S1", "S2"):
    ps = []
    for s in range(REPS):
        g = np.random.default_rng(10 * s); champ = Modelo(mundo(esc, 0, N_VENT, g)[0])
        for per in range(CAMBIO, P): ps.append(evaluar_challenger_libro(champ, None, mundo(esc, per, N_VENT, g)[0], Esquema)["promover"])
    prom[esc] = float(np.mean(ps))
show(pd.DataFrame({"fracción de periodos posteriores al cambio en que el libro PROMUEVE": prom}).T.round(2))

**Mira el resultado.** En S1 el listado promueve casi siempre. ¿Qué dice eso de «comparar poblaciones» como único chequeo?

**Respuesta:**

Que **las marginales no se movieron** (y que en S0, sin ningún cambio, el KS pide revisión humana en cerca del 8 % de los periodos: falsas alarmas) (en S1 solo cambió *cómo se asigna* el tratamiento) y la prueba estadística está fija en `True`: el listado promueve un modelo sesgado. Y en S2, donde sí cambió la población, no promueve: acierta, pero por la razón de siempre (KS), sin distinguir un cambio relevante de uno trivial. El capítulo lo dice con palabras («una revisión de que los supuestos de identificación siguen siendo defendibles»); el código no lo hace.



In [ ]:
# y el problema contrario: con ventanas grandes el KS alarma ante lo irrelevante
g = np.random.default_rng(1); a, b = g.normal(size=30000), g.normal(0.03, 1, 30000)
print(f"desplazamiento de 0,03 desviaciones, n=30000: p KS = {stats.ks_2samp(a, b).pvalue:.1e} → el libro diría «población no comparable» · ASMD = {asmd(a, b):.3f} · PSI = {psi(a, b):.4f} (ambos «estable»)")

## 4. Reescribir `evaluar_challenger`
Cuatro señales, todas calculables con lo que ya construimos: **soporte** (suficientes tratados y controles), **identificación vigente** (puntuar el propensity del champion en la ventana nueva, como en L25), **prueba estadística** (placebo de permutación con reajuste, como en L18) y **población comparable** con ASMD/PSI (el KS solo se reporta). Más una quinta: ¿el efecto **cambió** respecto al champion? (test z de 30.105).

In [ ]:
class Champion:
    def __init__(self, df):
        self.df = df.copy(); self.theta, self.se = estimar(df)
        self.prop = LogisticRegression(max_iter=1000).fit(df[COVS], df.D)
        self.auc_ref = auc_score(df.D, cross_val_predict(LogisticRegression(max_iter=1000), df[COVS], df.D, cv=5, method="predict_proba")[:, 1])
def señales(champ, ventana, B=15, seed=0):
    g = np.random.default_rng(seed); th, se = estimar(ventana)
    soporte = bool(min(ventana.D.sum(), (1 - ventana.D).sum()) >= 50)
    auc_rec = auc_score(ventana.D, champ.prop.predict_proba(ventana[COVS])[:, 1]); ident = bool(champ.auc_ref - auc_rec <= 0.10)
    plac = np.array([estimar(ventana.assign(D=g.permutation(ventana.D.to_numpy())))[0] for _ in range(B)])
    estad = bool(abs(plac.mean()) < 2 * plac.std(ddof=1) / np.sqrt(B) + 0.05)
    comp = bool(all(asmd(champ.df[c], ventana[c]) <= 0.25 and psi(champ.df[c], ventana[c]) <= 0.25 for c in COVS))
    cambio = estabilidad_efecto(champ.theta, champ.se, th, se)["alerta"]
    return {"soporte_suficiente": soporte, "identificacion_vigente": ident, "pasa_estadistica": estad, "poblacion_comparable": comp, "efecto_cambio": bool(cambio)}, th, se

## 5. Completa tú: `decide(señales)` y su precedencia
Las cinco señales pueden **chocar** (p. ej., la identificación se rompió *y* la población cambió). La función necesita un **orden de precedencia**, y ese orden es una decisión que se defiende por escrito. Devuelve una de: `abstenerse`, `revisar identificación`, `alertar`, `reentrenar`, `continuar`.

Piensa antes de escribir: ¿qué señal vuelve inútiles a las demás si falla? ¿cuál es la más grave aunque el número «se vea bien»? (La lectura del Cap. 30: el mecanismo de asignación ataca la identificación aunque el efecto aún no se haya movido.)

In [ ]:
# ✏️ Completa tú — decide
# Pista: Si no hay soporte, nada más importa: abstenerse. Después, la identificación. La población y la estadística piden revisión humana (alertar). Solo si todo pasa y el efecto cambió tiene sentido reentrenar.
def decide(s):
    # s: dict con soporte_suficiente, identificacion_vigente, pasa_estadistica, poblacion_comparable, efecto_cambio (bool)
    # Ordena las preguntas de la más a la menos grave y devuelve la primera que falla.
    raise NotImplementedError("completa decide")

In [ ]:
ok = dict(soporte_suficiente=True, identificacion_vigente=True, pasa_estadistica=True, poblacion_comparable=True, efecto_cambio=False)
c = lambda **kw: decide({**ok, **kw})
lab_check("decide", lambda: (c() == "continuar", c(efecto_cambio=True) == "reentrenar", c(identificacion_vigente=False) == "revisar identificación",
                             c(soporte_suficiente=False, identificacion_vigente=False) == "abstenerse", c(poblacion_comparable=False, efecto_cambio=True) == "alertar",
                             c(identificacion_vigente=False, poblacion_comparable=False) == "revisar identificación", c(pasa_estadistica=False, efecto_cambio=True) == "alertar"),
          [((), (True,) * 7, 0)])

**Mira el resultado.** Defiende el orden: ¿por qué «identificación» va antes que «población», y «población» antes que «efecto cambió»?

**Respuesta:**

**Soporte** primero: sin tratados y controles suficientes no hay estimador que juzgar. **Identificación** antes que población: si la no confusión ya no es defendible, el efecto estimado está mal *aunque* la población sea idéntica y *aunque* el efecto «se vea» estable (es el caso S1); la población que cambió, en cambio, puede dar un efecto perfectamente válido para otra población y solo requiere que un humano decida a cuál sirve. **Población y estadística** antes de «el efecto cambió»: un cambio de efecto que viene con un cambio de población o con un pipeline que falla no es una buena razón para reemplazar el modelo sin mirar. Solo con todo en orden un cambio de efecto justifica **reentrenar**. Otro orden podría defenderse (p. ej., estadística antes que identificación, porque un pipeline roto no merece ni la discusión de identificación); lo que no es defendible es no tener ninguno.



In [ ]:
def evaluar_challenger(champ, ventana, seed=0):
    s, th, se = señales(champ, ventana, seed=seed); d = decide(s)
    return {**s, "decision": d, "efecto_challenger": th, "se_challenger": se, "promover": d == "reentrenar",
            "p_ks_solo_informativo": {c: float(stats.ks_2samp(champ.df[c], ventana[c]).pvalue) for c in COVS}}

## 6. Reentrenar automáticamente frente a champion-challenger: error con verdad
Tres políticas por periodo: **automática** (el champion pasa a ser siempre el último challenger), **champion-challenger** (con `evaluar_challenger`) y **congelada** (nunca reentrena). Se mide el error del efecto **servido** tras el cambio.

In [ ]:
def corre(esc, politica, seed):
    g = np.random.default_rng(1000 + seed); champ = Champion(mundo(esc, 0, N_VENT, g)[0]); servido, dec, verdad = [], [], []
    for per in range(1, P):
        w, ate = mundo(esc, per, N_VENT, g)
        if politica == "automática": champ = Champion(w); d = "reentrenar"
        elif politica == "champion-challenger":
            r = evaluar_challenger(champ, w, seed=seed * 100 + per); d = r["decision"]
            if r["promover"]: champ = Champion(w)
        else: d = "continuar"
        if per >= CAMBIO: servido.append(champ.theta); dec.append(d); verdad.append(ate)
    return np.array(servido), dec, np.array(verdad)
tab, decs = {}, {}
for esc in ("S0", "S1", "S2"):
    for pol in ("automática", "champion-challenger", "congelada"):
        E_orig, E_nuevo, dd = [], [], []
        for s in range(REPS):
            sv, dec, vd = corre(esc, pol, s); E_orig.append(np.abs(sv - 2.0).mean()); E_nuevo.append(np.abs(sv - vd).mean()); dd += dec
        tab.setdefault(pol, {})[f"{esc}: error vs ATE original (2,0)"] = np.mean(E_orig)
        if esc == "S2": tab[pol][f"{esc}: error vs ATE de la población nueva"] = np.mean(E_nuevo)
        decs[(esc, pol)] = pd.Series(dd).value_counts(normalize=True).round(2).to_dict()
show(pd.DataFrame(tab).T.round(3)); print("decisiones del champion-challenger →", {k[0]: v for k, v in decs.items() if k[1] == "champion-challenger"})

**Mira el resultado.** Lee S1: ¿qué política es peor y por qué? Lee S2: ¿qué significa que la «congelada» y la «automática» tengan errores distintos según el blanco?

**Respuesta:**

**S1**: la automática es la peor, porque su error crece con cada reentrenamiento sobre datos con el confusor nuevo; el champion-challenger la frena en «revisar identificación» y se queda con el champion (su error es solo el ruido de estimación del periodo 0, igual que en S0). **S2**: no hay un «error» sin decir contra *qué población*. La automática sigue a la población nueva (error bajo contra el ATE nuevo, alto contra el original); la congelada es lo contrario. Reentrenar *cambia silenciosamente el estimando*. Por eso el champion-challenger responde «alertar»: no es un fallo del modelo, es una decisión que un humano debe tomar (¿servimos a la población nueva?).



### Señal de alerta: un cambio de población pasa el placebo
El capítulo lo dice: un cambio genuino y legítimo pasaría la prueba de placebo. Lo comprobamos en S2: la fracción de periodos posteriores al cambio en que cada señal da el visto bueno.

In [ ]:
res = {k: [] for k in ("soporte_suficiente", "identificacion_vigente", "pasa_estadistica", "poblacion_comparable")}
for s in range(REPS):
    g = np.random.default_rng(5000 + s); champ = Champion(mundo("S2", 0, N_VENT, g)[0])
    for per in range(CAMBIO, P):
        sg, _, _ = señales(champ, mundo("S2", per, N_VENT, g)[0], seed=s)
        for k in res: res[k].append(sg[k])
show(pd.DataFrame({"fracción de ventanas que pasan (S2, tras el cambio)": {k: np.mean(v) for k, v in res.items()}}).round(2))

**Mira el resultado.** ¿Qué señales dan «pasa» en S2 y cuál se enciende?

**Respuesta:**

El placebo (`pasa_estadistica`), el soporte y la identificación pasan: la población nueva es perfectamente estimable y su mecanismo de asignación no cambió. **Solo `poblacion_comparable` se enciende.** Si el control de promoción fuera solo el placebo, S2 se promovería sin que nadie lo notara.



## 7. Parte II — Un bandit con historia confundida (ejercicio 31.3.1-2)
Dos brazos. Una variable de contexto $U$ (no observada por el agente) afecta la recompensa, y un humano anterior eligió el brazo mirando $U$: usaba el **brazo 1** casi solo cuando $U$ era desfavorable. Resultado: la historia hace parecer **peor** al brazo 1 de lo que es.

$P(Y{=}1\mid a,U)$: brazo 0 → 0,40 / 0,70 ($U$=0/1); brazo 1 → 0,45 / 0,85. Recompensa **interventiva** (do): brazo 0 = 0,55; brazo 1 = 0,65 → el brazo 1 es el mejor.

In [ ]:
PY = {(0, 0): .40, (0, 1): .70, (1, 0): .45, (1, 1): .85}; DO = np.array([.5 * PY[(a, 0)] + .5 * PY[(a, 1)] for a in (0, 1)])
def historia(n, g, confundida=True):
    U = g.binomial(1, .5, n)
    a = np.where(U == 0, g.binomial(1, .9, n), g.binomial(1, .1, n)) if confundida else g.binomial(1, .5, n)
    y = g.binomial(1, np.vectorize(lambda aa, uu: PY[(aa, uu)])(a, U)); return a, y
g = np.random.default_rng(0); a_h, y_h = historia(20000, g)
obs = np.array([y_h[a_h == k].mean() for k in (0, 1)]); pa = np.array([(a_h == k).mean() for k in (0, 1)])
cotas = [(pa[k] * obs[k], pa[k] * obs[k] + (1 - pa[k])) for k in (0, 1)]
show(pd.DataFrame({"recompensa observacional P(Y|a)": obs, "recompensa interventiva (verdad)": DO, "cota inferior (Manski)": [c[0] for c in cotas], "cota superior (Manski)": [c[1] for c in cotas]}, index=["brazo 0", "brazo 1"]).round(3))

**Mira el resultado.** La historia dice que el brazo 0 es mucho mejor. ¿Qué dicen las cotas de Manski sobre qué tan seguro es eso?

**Respuesta:**

Observacionalmente, el brazo 0 (0,67) parece claramente mejor que el brazo 1 (0,49), pero las **cotas** —lo único que la historia garantiza sin supuestos sobre cómo se eligió— se solapan casi por completo: el rango posible del brazo 1 incluye valores muy por encima de los del brazo 0. La historia **no descarta** al brazo 1; solo lo hace *parecer* peor. Tratar los datos como si las decisiones pasadas hubieran sido aleatorias convierte esa apariencia en certeza.



In [ ]:
def bandit(politica, T, seed, confundida=True, N_H=500):
    g = np.random.default_rng(seed); a_h, y_h = historia(N_H, g, confundida)
    s1 = np.array([y_h[a_h == k].sum() for k in (0, 1)], float); n1 = np.array([(a_h == k).sum() for k in (0, 1)], float)
    pa = n1 / n1.sum(); obs = np.where(n1 > 0, s1 / np.maximum(n1, 1), .5); lo = pa * obs; hi = pa * obs + (1 - pa)
    if politica in ("arranque en frío", "con cotas"): al, be = np.ones(2), np.ones(2)     # la historia NO entra como muestras: a lo sumo como cotas
    else: al, be = 1 + s1, 1 + (n1 - s1)
    regret, elegido = 0.0, []
    for t in range(T):
        u = g.uniform(size=2)
        if politica == "con cotas":                      # la historia solo acota: se muestrea el posterior (solo con datos interventivos) dentro de [cota inferior, cota superior]
            cl, ch = stats.beta.cdf(lo, al, be), stats.beta.cdf(hi, al, be)
            theta = stats.beta.ppf(np.clip(cl + u * (ch - cl), 1e-9, 1 - 1e-9), al, be)
        else: theta = stats.beta.ppf(u, al, be)
        k = int(np.argmax(theta)); y = g.binomial(1, DO[k]); al[k] += y; be[k] += 1 - y; regret += DO.max() - DO[k]; elegido.append(k)
    return regret, float(np.mean(np.array(elegido[-T // 3:]) == 1)), (al[1] / (al[1] + be[1]))
Tb, nb = 1200, R(25, 120)
filas = {}
for nom, pol, conf in (("arranque en frío (ignora la historia)", "arranque en frío", True), ("historia como si fuera aleatoria", "historia ingenua", True),
                       ("historia solo como cotas (sin tomarla como muestras)", "con cotas", True), ("[control] historia ingenua pero SIN confusión", "historia ingenua", False)):
    r = np.array([bandit(pol, Tb, s, conf) for s in range(nb)]); filas[nom] = {"arrepentimiento acumulado": r[:, 0].mean(), "fracción final en el brazo 1": r[:, 1].mean(), "valor estimado del brazo 1 (verdad 0,65)": r[:, 2].mean()}
show(pd.DataFrame(filas).T.round(2))

**Mira el resultado.** ¿Por qué el agente con historia ingenua casi nunca llega al brazo 1, y qué muestran el control y la fila de las cotas?

**Respuesta:**

Su posterior para el brazo 1 arranca en torno a 0,49 con poca dispersión (cientos de observaciones) y para el brazo 0 en torno a 0,67: el muestreo de Thompson casi nunca prefiere al brazo 1, así que **no genera la evidencia que lo corregiría** — el sesgo se sostiene a sí mismo y el arrepentimiento crece linealmente. El control (misma historia con asignación pasada aleatoria) muestra que el problema no es usar la historia sino usarla **sin la confusión**: ahí ayuda. Usar la historia solo como cotas se comporta casi como el arranque en frío (aquí incluso algo peor): no sufre el sesgo, pero tampoco gana, porque las cotas de Manski con una historia así de confundida son anchas. La ventaja que el capítulo atribuye a combinar lo observacional con lo interventivo (cotas más estrechas, convergencia más rápida) viene de los resultados de Bareinboim et al., que **no** reproducimos aquí; esta simulación solo muestra el daño de ignorar la confusión y lo que ofrece la versión más simple de la alternativa.



## 8. Supuesto y bitácora

In [ ]:
s1_ok = señales(Champion(mundo("S1", 0, N_VENT, np.random.default_rng(3))[0]), mundo("S1", CAMBIO, N_VENT, np.random.default_rng(4))[0])[0]
fila_b = bitacora(
    pregunta="¿Puede promoverse el estimador reentrenado sin que una persona revise antes la identificación?",
    metodo="Champion-challenger con soporte, identificación (propensity viejo puntuado en la ventana), placebo con reajuste, ASMD/PSI y test z del efecto; precedencia explícita en decide()",
    estimando="El ATE que sirve el champion frente al del challenger",
    numero_ic="S1: la política automática hereda el sesgo del confusor no medido; el champion-challenger se queda con el champion y pide revisar identificación (ver tabla de §6)",
    supuesto=supuesto("no observable", "contrastado", "que el mecanismo de asignación sigue siendo el que el champion modeló: se contrasta puntuando el propensity viejo, pero un confusor no medido que no mueva las covariables observadas solo se ve por su efecto sobre la asignación"),
    decision=decide(s1_ok), lab="L27")

---
## Qué queda para el libro
Quedan en el libro: la teoría de bandits causales (Bareinboim, Forney y Pearl, 2015), los regímenes de tratamiento dinámicos y las fórmulas g de Robins. El bandit de la Parte II es una versión mínima propia, no una implementación de esos resultados.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*